# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-08_build_rag.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker — see the [course README](https://github.com/justkrismanohar/fine-tuning-build/blob/main/README.md).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day2/T1-08_build_rag.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day2"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day2" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)

# Day 2 helper lives here; Day 3 notebooks that import rag also add ../day2


# T1-08 Build RAG

**Time:** 10:45–12:00 &nbsp;|&nbsp; **Evidence:** source-to-answer trace + cited replies in `logs/` (live or labelled fallback)

**Notebook learning outcomes**

- Draw **retrieve, then generate** (RAG) as a pipeline, not a magic box.
- Run a classroom retriever (word overlap) and say how it stands in for embeddings.
- Send retrieved text through the Day 1 model client, or continue with a labelled local fallback.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — RAG in one page

**RAG** = Retrieval-Augmented Generation. Two steps, in order:

1. **Retrieve** — search an *approved* pile for chunks that match the question.
2. **Generate** — put those chunks into the prompt, then ask the model to answer **only** from that text (cite) or stop (abstain).

Day 1 generation had **no** handbook in the messages. The model continued text from weights. That is why it could invent Circular 99.

| Piece | What it is | What it is not |
|---|---|---|
| **Query** | The clerk’s question | A SQL statement (unless you chose **integration**) |
| **Retriever** | Scores chunks against the query | The generator |
| **k / top-k** | How many chunks you keep | “The whole EDMS” |
| **Score** | Overlap count here; cosine similarity if you use embeddings later | Proof the chunk is *allowed* |
| **Citation** | Handbook id + quoted span | A URL the model imagined |
| **Abstain** | “I cannot cite. Stop.” | A guessed circular |

**Embeddings (extra):** production RAG often turns chunks and the query into vectors and finds nearest neighbours. You are **not** training embeddings today. `civicdesk_rag.retrieve` counts shared words after dropping tiny stopwords. Same *job*: rank chunks. Different *engine*. If overlap feels crude, that is the point — you can see why it hit.

**Context window (from T1-02):** you retrieve *chunks* because a 400-page minute pack will not fit, and should not.

![RAG = retrieve, then generate](images/t2_rag_pipeline.jpg)


## Ministry link

The twelve chunks stand in for OPR notes, EDMS-style rules, circulars, ECMS, a KB article, minutes procedure — **fiction**.

| Question | Stands in for |
|---|---|
| Q-ATTACH overnight store | Finance / MOWI / RDLG |
| Q-C2 library-studies diploma | Labour / Health (quote C2, do not hire) |
| Q-PWD reset my password | Trade IMSD (KB says the assistant must not reset) |
| Q-MISSING Circular 88 drones | Abstain drill for everyone |
| Q-CAB coastal-security minute | Finance access (T1-10) |

This lab is **N3** with **N1** and **N8** (quote, do not write a long unsourced summary).


## 0. Backend — demo, OLMo, or Phi

Retrieval always runs on this machine. This cell only chooses **what generates**.

| `BACKEND` | Use when |
|---|---|
| **`demo`** | No GPU, overnight, or the server is down. Canned / template replies. **Not** a live-model score. |
| **`olmo`** | Local OLMo is up (T1-01a / `docker compose`, port **8000**). Needs VRAM. |
| **`phi`** | Local Phi is up (port **8001**). Compose loads OLMo only unless you set `CIVICDESK_LOAD_MODELS=olmo,phi`. |

Change the one marked line and re-run **this** cell. Later generate cells in this kernel follow it. If the live call fails, the notebook continues with a labelled local fallback.


In [ ]:
# [CHANGE] where generate() goes. Retrieval still runs here either way.
BACKEND = "demo"  # "demo" | "olmo" | "phi"

import civicdesk_rag as rag

rag.set_backend(BACKEND)


## 1. The questions


In [ ]:
# [RUN]
import civicdesk_rag as rag

rag.print_questions()


In [ ]:
# [RUN] Check the model you're about to use.
import civicdesk_rag as rag
lab = rag.import_lab()
cfg = rag.endpoint_probe()
print("backend:", cfg["backend"], "mode:", cfg["mode"], "model:", cfg["model"])
print("endpoint:", cfg.get("olmo_url"))
print("timeouts:", cfg["timeouts"])
print(cfg["note"])
print(cfg["local_vs_host"])
if cfg.get("offline"):
    print(cfg["offline"])
print("A health check is not a successful generation request.")


## From a source file to an answer (~20 minutes)

This exercise uses **one fictional policy file**, not `handbooks.json`. You will see the file, the chunks, the in-memory keyword index, the prompt, then a generation request. Embeddings are another retrieval engine; they are not running in the background. The handbook evaluation corpus stays untouched.


In [ ]:
# [RUN]
from pathlib import Path
import json
import civicdesk_rag as rag

SOURCE_PATH = Path("data/source_documents/CD-PRINT-01.md")  # UTF-8 text, required offline path
doc = rag.load_source_document(SOURCE_PATH)
print("SOURCE_PATH:", doc["path"])
print("title:", doc["title"], "version:", doc["version"])
print("extraction:", doc["extraction"])
print("--- original contents ---")
print(doc["text"])
print("--- intake metadata ---")
print(json.dumps(doc["meta"], indent=2))


### Read and extract

This source is a Markdown file. Extraction is **reading text**, not OCR. Scanned images need a checked OCR step first (T1-07). Nothing here pretends a live OCR engine ran.


In [ ]:
# [CHANGE] chunk size. Rebuild after you change it.
MAX_WORDS = 40
OVERLAP = 0

import civicdesk_rag as rag
from pathlib import Path

doc = rag.load_source_document(Path("data/source_documents/CD-PRINT-01.md"))
SRC_CHUNKS = rag.chunk_source(doc, max_words=MAX_WORDS, overlap=OVERLAP)
for c in SRC_CHUNKS:
    print(f"{c['id']} seq={c['sequence']} ver={c['version']} path={c['source_path']}")
    print(c["text"])
    print("clearance", c["clearance"], "index_ok", c["index_ok"], "owner", c["owner"])
    print()

alt = 20 if MAX_WORDS != 20 else 60
other = rag.chunk_source(doc, max_words=alt, overlap=OVERLAP)
print("boundaries at", MAX_WORDS, [(c["id"], len(c["text"].split())) for c in SRC_CHUNKS])
print("boundaries at", alt, [(c["id"], len(c["text"].split())) for c in other])
print("changed boundaries vs max_words", alt)


In [ ]:
# [RUN] in-memory keyword index from THESE chunks only.
import civicdesk_rag as rag

SRC_INDEX = rag.build_keyword_index(SRC_CHUNKS)
rag.preview_index(SRC_INDEX)
print("This is not the handbook JSON.")


In [ ]:
# [CHANGE] a question whose evidence is in CD-PRINT-01.
SRC_QUESTION = "May a clerk print a leave letter at home overnight?"
ROLE = "clerk"

import json
import civicdesk_rag as rag

hits = rag.query_index(SRC_INDEX, SRC_QUESTION, k=5, min_score=1)
print("hits from the exercise index (not handbooks.json):")
print(json.dumps([{"id": h["id"], "score": h["score"], "source": h.get("source_path")} for h in hits], indent=2))
visible, blocked = rag._gate_hits(hits, ROLE)
print("visible", [v["id"] for v in visible], "blocked", [(b["id"], b["reason"]) for b in blocked])
SRC_VISIBLE = visible


In [ ]:
# [RUN] inspect the actual messages before any backend call.
import civicdesk_rag as rag

SRC_MESSAGES = rag.build_messages(SRC_QUESTION, SRC_VISIBLE)
for m in SRC_MESSAGES:
    print("=" * 40, m["role"])
    print(m["content"])
print()
n = rag.prompt_token_count(SRC_MESSAGES)
print("prompt token count (Day 1 tokenizer if present):", n)
print("The document supplied context. It did not add weights.")


In [ ]:
# [RUN] send those messages. Demo/offline continues with the local cite-or-abstain helper.
import civicdesk_rag as rag

src_out = rag.generate_with_rag(SRC_QUESTION, role=ROLE, pool=SRC_CHUNKS, min_score=1, k=5)
print("mode:", src_out["mode"], "chat_mode:", src_out["chat_mode"], "evidence:", src_out["evidence"])
print("generation_succeeded:", src_out["generation_succeeded"])
print("messages_status:", src_out["messages_status"])
print(src_out["reply"])
print("checks:", src_out["checks"])
print("trace:", SOURCE_PATH.name, "->", [c["id"] for c in SRC_CHUNKS], "-> hits", src_out["hits"], "->", src_out["mode"])
rag.log_record(src_out, name="T1-08_source")


In [ ]:
# [RUN] prove the connection: drop the document, rebuild, restore.
import civicdesk_rag as rag

empty = rag.build_keyword_index([])
miss = rag.query_index(empty, SRC_QUESTION, k=5, min_score=1)
print("after exclude, hits:", [h["id"] for h in miss])
msgs = rag.build_messages(SRC_QUESTION, [])
print("HB-PRINT chunks in messages:", "CD-PRINT" in msgs[1]["content"])
restored = rag.query_index(SRC_INDEX, SRC_QUESTION, k=5, min_score=1)
print("after restore, hits:", [h["id"] for h in restored])
print("An answer may still be guessed from model weights. Judge the provenance assertion, not a promised wording change.")
empty_out = rag.generate_with_rag(SRC_QUESTION, role=ROLE, pool=[], min_score=1, k=5)
print("after exclude, message has CD-PRINT:", "CD-PRINT" in empty_out["messages"][1]["content"])
print("after exclude, hits:", empty_out["hits"])


## 2. Retrieve on the frozen handbook corpus

Watch ids and scores. A score is **shared tokens**, not permission. This corpus is separate from the print-rule exercise above.


In [ ]:
# [CHANGE] try Q-C2 or Q-PWD after you run Q-ATTACH. Then run the paraphrase.
QUESTION_ID = "Q-ATTACH"
PARAPHRASE = "Can I save uploaded files from a ticket overnight on my laptop?"

import json
import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}[QUESTION_ID]
print("USER:", q["user"])
print("expect_chunk:", q["expect_chunk"])
hits = rag.retrieve(q["user"], k=3)
print("top-k hits")
print(json.dumps([{"id": h["id"], "score": h["score"], "overlap": h["overlap"]} for h in hits], indent=2))
print()
print("PARAPHRASE:", PARAPHRASE)
ph = rag.retrieve(PARAPHRASE, k=3)
print("paraphrase hits:", [{"id": h["id"], "score": h["score"]} for h in ph] or "none — keyword retriever abstains")


In [ ]:
# [OPTIONAL] Meaning search (precomputed scores only; no vector database in this room).
from pathlib import Path
import json

p = Path("data/embedding_scores.json")
if not p.is_file():
    print("embedding_scores.json is not in this package (image extra not approved). A cosine score is still not permission.")
else:
    scores = json.loads(p.read_text(encoding="utf-8"))
    print("paraphrase top:", scores.get("Q-PARAPHRASE"))
    print("drone / missing top:", scores.get("Q-MISSING"))
    print("A high score on HB-CAB-X is not permission to show it to a clerk.")


In [ ]:
# [CHANGE] Where the chunk ends. Run twice: 25 then 60.
MAX_WORDS = 25

import civicdesk_rag as rag

c2 = next(c for c in rag.load_chunks() if c["id"] == "HB-VAC-C2")
parts = rag.chunk_text(c2["text"], max_words=MAX_WORDS, overlap=0)
print("MAX_WORDS", MAX_WORDS, "n_chunks", len(parts))
for i, p in enumerate(parts, 1):
    print(f"--- {i} ---")
    print(p)
print("At 25 words the human-review sentence often sits in a later chunk than the C2 rule.")


### Teaching note

If `HB-ATTACH-04` is on top for Q-ATTACH, the overlap retriever did its job. If a different internal chunk wins, read `overlap`.

**k=3** means you only *look* at three. A score is not permission. Embeddings would be a different engine for the same job.

The model only sees what is in the next prompt. That is the Day 1 spotlight again: later tokens attend to what you put in the window. We will not re-teach attention here.


## 3. Generate with the retrieved text

First look at the messages. Then send them through the Day 1 client. If the backend is demo or down, the notebook continues with the local cite-or-abstain helper and labels that output as **not a model answer, not scored**.


In [ ]:
# [RUN]
import json
import civicdesk_rag as rag

qs = {x["id"]: x for x in rag.load_questions()}
q = qs["Q-ATTACH"]
hits = rag.retrieve(q["user"], k=3)
visible, blocked = rag._gate_hits(hits, "clerk")
msgs = rag.build_messages(q["user"], visible)
print("=== messages the model would receive for Q-ATTACH ===")
for m in msgs:
    print(m["role"].upper())
    print(m["content"][:2000])
print("token count:", rag.prompt_token_count(msgs))
print("blocked:", [(b["id"], b["reason"]) for b in blocked])
print()

for qid in ("Q-ATTACH", "Q-C2", "Q-PWD"):
    q = qs[qid]
    out = rag.generate_with_rag(q["user"], role="clerk")
    print("=" * 60)
    print(qid, "mode", out["mode"], "evidence", out["evidence"], "visible", out["visible_ids"])
    print(out["reply"])
    print("checks:", {k: out["checks"][k] for k in ("cited_ids", "unknown_citations", "abstained", "outcome_met") if k in out["checks"]})
    rag.log_record(out, name=f"T1-08_{qid}")


### After you run

Do not expect a particular sentence from the model. Name any **unknown citation**, **altered quote**, or **wrong NOT IN SOURCES** you see. Counts belong in T1-09.

**Write-up:** copy one quoted span from a live or local reply and circle the citation id. If you cannot point at a span, you do not have RAG.


In [ ]:
# [OPTIONAL] deep-code: implement overlap yourself and compare to the helper
import civicdesk_rag as rag

q = "overnight attachments file store"
mine = rag.retrieve(q, k=5)
print("query tokens:", sorted(rag._tok(q)))
for h in mine:
    print(h["id"], h["score"], h["overlap"])
